Funció per a poder llegir .json complexes

In [2]:
import json

def leer_json(file_path):
    data = []
    with open(file_path, 'r') as file:
        
        for line in file:
            try:
                json_obj = json.loads(line)
                data.append(json_obj)
            except json.JSONDecodeError as e:
                print(f"Error al decodificar JSON: {e}")
    return data


## Generació de books.csv i users.csv
La informació s'extreu a partir dels arxius 'short_interactions.csv' i 'goodreads_book_genres_initial.json'

In [4]:
!pip install polars
import polars as pl
libros_leidos = pl.read_csv('short_interactions.csv') 

generos_libros = leer_json('goodreads_book_genres_initial.json')
generos_libros = pl.DataFrame(generos_libros)


  Obtaining dependency information for polars from https://files.pythonhosted.org/packages/26/84/7fe894b055e345e8f9c356e107cb48f5f3ce9e729804589bec43382e429f/polars-0.19.18-cp38-abi3-win_amd64.whl.metadata
   ---------------------------------------- 0.0/24.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.8 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.8 MB 262.6 kB/s eta 0:01:35
   ---------------------------------------- 0.1/24.8 MB 393.8 kB/s eta 0:01:03
   ---------------------------------------- 0.1/24.8 MB 711.9 kB/s eta 0:00:35
   ---------------------------------------- 0.3/24.8 MB 1.3 MB/s eta 0:00:20
    --------------------------------------- 0.5/24.8 MB 1.8 MB/s eta 0:00:14
   - -------------------------------------- 0.7/24.8 MB 2.3 MB/s eta 0:00:11
   - -------------------------------------- 0.

In [5]:
# Nos quedamos solo con los libros leidos por + de 10 personas
conteo_libros = libros_leidos.group_by(libros_leidos.columns[1]).agg(pl.count())
X = 10
libros_leidos = libros_leidos.filter(libros_leidos[libros_leidos.columns[1]].is_in(conteo_libros.filter(pl.col('count') > X)[libros_leidos.columns[1]]))

# Lista de 'book_id' del DataFrame libros_leidos
libros_leidos = libros_leidos.with_columns(libros_leidos['book_id'].cast(pl.Utf8))
libros_leidos_list = libros_leidos['book_id'].unique().to_list()

# Filtrar generos_libros para mantener solo aquellos libros que estén en book_ids_filtrados
generos_libros = generos_libros.with_columns(generos_libros['book_id'].cast(pl.Utf8))
generos_libros = generos_libros.filter(generos_libros['book_id'].is_in(libros_leidos_list))

#convertir la columna 'generos' en una columna para cada genero
generes = []
for i in generos_libros['genres']:
    generes.append(list(i.values()))

generes = list(zip(*generes))

noms_generes = ['fantasy, paranormal', 'fiction', 'mystery, thriller, crime', 'poetry', 
           'history, historical fiction, biography', 'romance', 'non-fiction', 
           'children', 'young-adult', 'comics, graphic']


# Añadir las columnas al DataFrame
for lista, nombre in zip(generes, noms_generes):
    generos_libros = generos_libros.with_columns(pl.Series(nombre, lista))

generos_libros = generos_libros.drop('genres')

for columna in generos_libros.columns:
    generos_libros = generos_libros.with_columns(generos_libros[columna].fill_null(0))

A partir de la base de dades de llibres, fem un nou dataset on s'agrupa per usuaris, i dona informació de:
- La quantiat de llibres que ha llegit
- El llistat de llibres que ha llegit 
- La puntuació mitja del llibres que fa
- El percentatge de llibres que ha fet un review

In [6]:
df_agrupado = libros_leidos.group_by('user_id').agg([
    pl.col('book_id').alias('libros_leidos'),
    pl.col('is_read').sum().alias('num_leidos'),
    pl.col('rating').mean().alias('mean_rating'),
    (pl.col('is_reviewed').sum() / pl.count()).alias('prop_reviwed')
])

In [7]:
#es converteix la columna 'libros_leidos'  a una string amb tots els llibres separats per un espai
#això es fa perque sino no es pot guardar el CSV
df_agrupado = df_agrupado.with_columns(
    df_agrupado['libros_leidos'].map_elements(lambda lst: " ".join(lst)).alias('read_books')
)
df_agrupado = df_agrupado.drop('libros_leidos')


## Base de dades de llibres

Ens quedem només amb els llibres que han llegit els usuaris

In [8]:
#Com l'arxiu és molt gran, s'ha dividit en 25 arxius per poder llegir-los
#vas arxiu per arxiu mirant quina llibres es troben en la base de llibres dels users i et quedes només amb aquests
dfs= []
for i in range(1, 26):
    libros = leer_json(f'books_parte_{i}.json')
    libros = pl.DataFrame(libros)
    libros = libros.with_columns(libros['book_id'].cast(pl.Utf8))
    libros = libros.filter(libros['book_id'].is_in(libros_leidos_list))
    dfs.append(libros)

#ajuntes tots els fitxers en 1
llibres = pl.concat(dfs)

Mirem que té 'llibres'

In [9]:
llibres

isbn,text_reviews_count,series,country_code,language_code,popular_shelves,asin,is_ebook,average_rating,kindle_asin,similar_books,description,format,link,authors,publisher,num_pages,publication_day,isbn13,publication_month,edition_information,publication_year,url,image_url,book_id,ratings_count,work_id,title,title_without_series
str,str,list[str],str,str,list[struct[2]],str,str,str,str,list[str],str,str,str,list[struct[2]],str,str,str,str,str,str,str,str,str,str,str,str,str,str
"""0850308712""","""5""",[],"""US""","""""","[{""32"",""to-read""}, {""3"",""runes""}, … {""1"",""magick-and-myth""}]","""""","""false""","""3.40""","""""",[],"""""","""""","""https://www.go…","[{""149918"",""""}]","""""","""""","""""","""9780850308716""","""""","""""","""""","""https://www.go…","""https://images…","""287140""","""15""","""278577""","""Runic Astrolog…","""Runic Astrolog…"
"""037583687X""","""615""",[],"""US""","""""","[{""4248"",""to-read""}, {""258"",""historical-fiction""}, … {""4"",""newberys""}]","""""","""false""","""3.98""","""B0010SEMV4""","[""614054"", ""272343"", … ""1833502""]","""It's 1953 and …","""Hardcover""","""https://www.go…","[{""137561"",""""}]","""Random House B…","""288""","""25""","""9780375836879""","""7""","""""","""2006""","""https://www.go…","""https://images…","""89377""","""6949""","""86258""","""Penny from Hea…","""Penny from Hea…"
"""0590417010""","""193""",[],"""US""","""eng""","[{""450"",""to-read""}, {""64"",""picture-books""}, … {""1"",""k-2""}]","""""","""false""","""4.43""","""B017RORXNI""","[""834493"", ""452189"", … ""1081530""]","""In Newbery Med…","""Hardcover""","""https://www.go…","[{""5411"",""""}]","""Blue Sky Press…","""40""","""1""","""9780590417013""","""9""","""""","""1995""","""https://www.go…","""https://images…","""89378""","""1331""","""86259""","""Dog Heaven""","""Dog Heaven"""
"""0865165149""","""1""",[],"""US""","""""","[{""4"",""to-read""}, {""1"",""owned""}]","""""","""false""","""4.50""","""""",[],"""A unique tool …","""""","""https://www.go…","[{""166520"",""""}, {""166517"",""Editor""}, … {""3958429"",""Translator""}]","""Bolchazy-Cardu…","""""","""1""","""9780865165144""","""9""","""""","""2002""","""https://www.go…","""https://s.gr-a…","""287142""","""2""","""23097481""","""Servius' Comme…","""Servius' Comme…"
"""0553576348""","""41""",[],"""US""","""""","[{""487"",""to-read""}, {""98"",""currently-reading""}, … {""1"",""5-stars-favorite""}]","""""","""false""","""4.09""","""B008WOUJOI""","[""644540"", ""2309183"", … ""2168992""]","""""","""""","""https://www.go…","[{""51308"",""""}, {""51307"",""""}]","""""","""""","""""","""9780553576344""","""""","""""","""""","""https://www.go…","""https://images…","""89379""","""627""","""86260""","""Hello from Hea…","""Hello from Hea…"
"""""","""7""","[""158860"", ""1100745""]","""US""","""eng""","[{""12796"",""to-read""}, {""972"",""classics""}, … {""8"",""public-domain""}]","""""","""false""","""3.88""","""""","[""3311228"", ""913719"", … ""722035""]","""In 1888 Lord a…","""Paperback""","""https://www.go…","[{""10885"",""""}]","""Ballantine Boo…","""245""","""""","""0345227018095""","""""","""""","""""","""https://www.go…","""https://s.gr-a…","""828189""","""64""","""1774048""","""Tarzan of the …","""Tarzan of the …"
"""0749729120""","""1""","[""432901""]","""US""","""""","[{""33"",""to-read""}, {""4"",""enid-blyton""}, … {""1"",""english""}]","""""","""false""","""3.66""","""""",[],"""Mike, Belinda …","""""","""https://www.go…","[{""10657"",""""}]","""""","""93""","""""","""9780749729127""","""""","""""","""""","""https://www.go…","""https://s.gr-a…","""388676""","""59""","""2055963""","""The Queen Eliz…","""The Queen Eliz…"
"""0380821001""","""61""",[],"""US""","""""","[{""550"",""to-read""}, {""70"",""science-fiction""}, … {""1"",""backlog-ebook""}]","""""","""false""","""3.81""","""B002QHATGA""","[""114555"", ""219231"", … ""622100""]","""The ""magic"" th…","""Paperback""","""https://www.go…","[{""20560"",""""}]","""Harper Voyager…

##### Variables eliminades:
- text_reviws_count -> quantitat de reviws que ha rebut el llibre per part dels ususaris
- series -> indica si a part del llibre s'ha generat una serie o peli a partir daquest
- popular_shelves -> a quins apartats de good read pertanyia
- asin -> codi asin del llibre, per reconeixe'l
- is_ebook -> si el llibre es troba en format ebook o no
- kindle_asin -> codi asin del llibre en format kindle, per reconeixe'l
- description -> descripció llarga sobre de que tracta el llibre
- link -> link al llibre 
- publisher -> la editorial del llibre
- publication_day -> dia en que es va publicar el llibre
- publication_month -> mes en que es va publicar el llibre
- edition_information -> informació adicional sobre la edició del llibre (pràcticament totes nul)
- url -> url per poder trobar el llibre
- image_url url per poder trobar la portada del llibre
- work_id -> identificador de obres (més general que els llibres)
- title_without_series -> títol a seques del llibre

Variables a estudiar:
- isbn -> conté els ibsn de 10 dígits dels llibres
-  country_code -> la nacionalitat d'on s'han extret les dades
- language_code -> en quin idioma està escrit el llibre original
- average_rating -> mitja de la puntuació al llibre
- similar_books -> aquells llibres de la base de dades que es considera que estan realcionats
- formal -> tipus de llibre (tapadura, blanda, de bolsillo...)
- authors -> autors que han creat el llibre (pot ser autor, ilustradors, autors de l'apendix....)
- num_pages -> pàgines del llibre
- publication_year -> any de la publicació del llibre
- isbn13 -> conté els ibsn de 13 dígits dels llibres
- book_id -> identificador del llibre
- ratings_count -> quantitat de persones que l'han puntuat
- title -> titol del llibre. Si pertany a una saga o serie també inclou el nom de la saga




In [156]:
#Columnes a borrar
no_util = ['url','image_url','series', 'asin', 'is_ebook', 'kindle_asin', 'description', 'link', 'publication_day', 'edition_information', 'title_without_series', 'publication_month', 'publisher', 'text_reviews_count', 'popular_shelves', 'work_id']
llibres = llibres.drop(no_util)

Preprocessament

In [157]:
llibres

isbn,country_code,language_code,average_rating,similar_books,format,authors,num_pages,isbn13,publication_month,publication_year,book_id,ratings_count,work_id,title
str,str,str,str,list[str],str,list[struct[2]],str,str,str,str,str,str,str,str
"""0850308712""","""US""","""""","""3.40""",[],"""""","[{""149918"",""""}]","""""","""9780850308716""","""""","""""","""287140""","""15""","""278577""","""Runic Astrolog…"
"""037583687X""","""US""","""""","""3.98""","[""614054"", ""272343"", … ""1833502""]","""Hardcover""","[{""137561"",""""}]","""288""","""9780375836879""","""7""","""2006""","""89377""","""6949""","""86258""","""Penny from Hea…"
"""0590417010""","""US""","""eng""","""4.43""","[""834493"", ""452189"", … ""1081530""]","""Hardcover""","[{""5411"",""""}]","""40""","""9780590417013""","""9""","""1995""","""89378""","""1331""","""86259""","""Dog Heaven"""
"""0865165149""","""US""","""""","""4.50""",[],"""""","[{""166520"",""""}, {""166517"",""Editor""}, … {""3958429"",""Translator""}]","""""","""9780865165144""","""9""","""2002""","""287142""","""2""","""23097481""","""Servius' Comme…"
"""0553576348""","""US""","""""","""4.09""","[""644540"", ""2309183"", … ""2168992""]","""""","[{""51308"",""""}, {""51307"",""""}]","""""","""9780553576344""","""""","""""","""89379""","""627""","""86260""","""Hello from Hea…"
"""""","""US""","""eng""","""3.88""","[""3311228"", ""913719"", … ""722035""]","""Paperback""","[{""10885"",""""}]","""245""","""0345227018095""","""""","""""","""828189""","""64""","""1774048""","""Tarzan of the …"
"""0749729120""","""US""","""""","""3.66""",[],"""""","[{""10657"",""""}]","""93""","""9780749729127""","""""","""""","""388676""","""59""","""2055963""","""The Queen Eliz…"
"""0380821001""","""US""","""""","""3.81""","[""114555"", ""219231"", … ""622100""]","""Paperback""","[{""20560"",""""}]","""512""","""9780380821006""","""4""","""2003""","""89588""","""1480""","""822875""","""The Visitor"""
"""0953626954""","""US""","""en-GB""","""3.67""",[],"""Paperback""","[{""150591"",""""}]","""248""","""9780953626953""","""""","""2001""","""257654""","""27""","""249714""","""Gillian of the…"


In [158]:
#Authors ens quedem amb el primer, que és l'escriptor
autors = []
for author in llibres['authors']:
    if len(author) == 0:
        autors.append(None)
    else:
        autors.append(author[0]['author_id'])
      
llibres = llibres.with_columns(pl.Series('author_id', autors))
llibres = llibres.drop('authors')

In [159]:
#Diferències entre 'country_code' i 'language_code'
print('Num Country Code -> ',llibres['language_code'].unique().shape[0])
print('Num Laungage Code -> ' ,llibres['country_code'].unique().shape[0])
llibres = llibres.drop('country_code')

Num Country Code ->  54
Num Laungage Code ->  1


In [160]:
#Diferencies entre 'isbn' i 'isbn13'
print('No isbn13 -> ',llibres.filter(llibres['isbn13'] == "").shape[0])
print('No isbn ->', llibres.filter(llibres['isbn'] == "").shape[0])

#reemplaçar isbn13 d'aquells que no tenen per isbn
llibres = llibres.with_columns(
    pl.when(llibres['isbn13'] == "")
    .then(llibres['isbn'])
    .otherwise(llibres['isbn13'])
    .alias('isbn13')
)

print('No isbn amb el preprocessat ->', llibres.filter(llibres['isbn13'] == "").shape[0])
llibres = llibres.drop('isbn')

No isbn13 ->  644
No isbn -> 705


In [ ]:
#que els llibres recomanats siguin únicament dels que es troben a la base de dades
llibres_ids = llibres['book_id'].unique().to_list()

autors_sim = []
for similars in llibres['similar_books']:
    if len(similars) == 0:
        autors_sim.append(None)
    else:
        new = []
        for llibre in similars:
            if llibre in llibres_ids: new.append(llibre)
        autors_sim.append(new)

llibres = llibres.with_columns(pl.Series('similar_books', autors_sim))

#Converteix la llista de llibres comuns en una string amb tots els llibres comuns separats per un espai
#Es fa això perque sino no es pot guardar la base de dades
llibres = llibres.with_columns(
    llibres['similar_books'].apply(lambda lst: " ".join(lst)).alias('similar_books')
)

In [213]:
#Convertir els "" en Nones
for variable in llibres.columns:
    llibres = llibres.with_columns(
    pl.when(llibres[variable] == "")
    .then(None)
    .otherwise(llibres[variable])
    .alias(variable)
)

In [232]:
#Canviar average_rating i rating_count a numèriques
llibres = llibres.with_columns(llibres['average_rating'].cast(pl.Float64))
llibres = llibres.with_columns(llibres['ratings_count'].cast(pl.Int64))
llibres = llibres.with_columns(llibres['num_pages'].cast(pl.Int64))

In [ ]:
# Filtrar generos_libros para mantener solo aquellos libros que estén en llibres
generos_libros = generos_libros.filter(generos_libros['book_id'].is_in(llibres_ids))
print(generos_libros.shape,llibres.shape)

#Ajuntem els 2 dataframes
llibres = llibres.join(generos_libros, on='book_id', how='inner')

In [238]:
llibres

language_code,average_rating,similar_books,format,num_pages,isbn13,publication_year,book_id,ratings_count,title,author_id,"fantasy, paranormal",fiction,"mystery, thriller, crime",poetry,"history, historical fiction, biography",romance,non-fiction,children,young-adult,"comics, graphic"
str,f64,str,str,i64,str,str,str,i64,str,str,i64,i64,i64,i64,i64,i64,i64,i64,i64,i64
null,3.4,null,null,null,"""9780850308716""",null,"""287140""",15,"""Runic Astrolog…","""149918""",0,0,0,0,0,0,3,0,0,0
null,3.98,"""199623 375749""","""Hardcover""",288,"""9780375836879""","""2006""","""89377""",6949,"""Penny from Hea…","""137561""",0,425,0,0,330,0,0,190,93,0
"""eng""",4.43,"""140185 197998 …","""Hardcover""",40,"""9780590417013""","""1995""","""89378""",1331,"""Dog Heaven""","""5411""",0,13,0,0,0,0,2,109,1,0
null,4.5,null,null,null,"""9780865165144""","""2002""","""287142""",2,"""Servius' Comme…","""166520""",0,0,0,0,0,0,0,0,0,0
null,4.09,"""46609 416792""",null,null,"""9780553576344""",null,"""89379""",627,"""Hello from Hea…","""51308""",6,0,1,0,0,0,19,0,0,0
"""eng""",3.88,"""83169 107284 3…","""Paperback""",245,"""0345227018095""",null,"""828189""",64,"""Tarzan of the …","""10885""",507,825,11,0,64,45,0,31,36,0
null,3.66,null,null,93,"""9780749729127""",null,"""388676""",59,"""The Queen Eliz…","""10657""",0,2,0,0,0,0,0,8,0,0
null,3.81,"""219231 455901 …","""Paperback""",512,"""9780380821006""","""2003""","""89588""",1480,"""The Visitor""","""20560""",86,130,1,0,0,0,0,0,0,0
"""en-GB""",3.67,null,"""Paperback""",248,"""9780953626953""","""2001""","""257654""",27,"""Gillian of the…","""150591""",0,3,0,0,0,0,0,2,0,0


In [239]:
llibres.write_csv('books.csv')

In [ ]:
libros_leidos = libros_leidos.filter(libros_leidos['book_id'].is_in(llibres_ids))
libros_leidos.write_csv('users.csv')